In [1]:
## setting environment 
! pip install llmcompressor # latest version of llm-compressor 

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B" # 0.6B model 
OUTPUT_DIR = "Outputs"
DATASET_NAME = "wikitext-2-raw-v1"

print(f"Base model:      {MODEL_NAME}")
print(f"Quantized model: {OUTPUT_DIR}")

Base model:      Qwen/Qwen3-0.6B
Quantized model: Outputs


In [3]:
## choosen activation => W8A16 => weights 8bit and activations bfloat16/float16 (Alternative)
### recomended compressrion scheme (Turning Hardware) => W8A8
## compression algorithm (non MOE models)
##   Comparasion 
## 1. AWQ (Activation-aware Weighted Quantization)
## 2. GPTQ (Generative Pretrain )
## 3. RTN (Round to nearest)
## Compressrion algorithm (MOE model) => REAp 

## GPTQ Modifier 

In [20]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme = "W8A8", # weight int 8 ,, activations int 8 => for T4
    targets = "Linear", ## only quantize the linear layers 
    ignore = ["lm_head"]
)
print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W8A8' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False requires_calibration_data=True index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False block_size=128 dampening_frac=0.01 actorder=static batched_quantization='auto'


In [48]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [13]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
raw_dataset = load_dataset(
    "Salesforce/wikitext", 
    "wikitext-2-raw-v1", 
    split="train"
)

# Filter empty lines
raw_dataset = raw_dataset.filter(lambda x: len(x["text"].strip()) > 0)

raw_dataset

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 23767
})

In [17]:
raw_dataset["text"][0]

' = Valkyria Chronicles III = \n'

In [18]:
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=2048,  # Adjust seq length according to your VRAM
        return_tensors=None,
    )

tokenized_dataset = raw_dataset.map(
    tokenize_fn, 
    batched=True, 
    remove_columns=raw_dataset.column_names
)

# 4. Enforce integer dtypes (int64 / LongTensor)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

In [35]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 23767
})

### the above optimization is required => because for quantization the tensor expecting are not in int8 or float but in cuda.float16 so we have to convert then to it 

In [21]:
from llmcompressor import oneshot

if not os.path.isdir(f"kaggle/working/{OUTPUT_DIR}"):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset=tokenized_dataset,
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=4096,
        num_calibration_samples=256,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

2026-10-23T04:41:10.2563 | reset | INFO - Compression lifecycle reset
2026-10-23T04:41:10.2627 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 196/196 [00:00<00:00, 7202.29it/s]

2026-10-23T04:41:10.3024 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-10-23T04:41:10.3031 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



(2/29): Calibrating: 100%|██████████| 256/256 [00:01<00:00, 156.66it/s]


...

2026-10-23T04:42:29.0216 | _retie_embeddings | INFO - Re-tied input/output embeddings; saving a single shared table.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-23T04:42:29.4095 | resave_config | INFO - Resaved original config with patched fields to Outputs/config.json


Dispatching model: 100%|██████████| 427/427 [00:00<00:00, 18438.30it/s]

Quantization complete. Model saved to: Outputs


In [24]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [26]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [28]:
## comparsion for GPTQ quantization with original 
def format_size(nbytes):
    if nbytes < 1024**2: # (1024**2 = 1mb) if less < 1024**2 then its kb 
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3: # (1024**3 = 1gb) if less < 1024**3 then its gb 
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig = model.get_memory_footprint()
size_q = folder_size(OUTPUT_DIR)
reduction = (1 - size_q / size_orig) * 100 if size_orig > 0 else 0

print("Model Size Comparison")
print("=" * 45)
print(f"Original (BF16):    {format_size(size_orig)}")
print(f"Quantized (W4A16):  {format_size(size_q)}")
print(f"Reduction:          {reduction:.0f}%")

Model Size Comparison
Original (BF16):    1.11 GB
Quantized (W4A16):  728.5 MB
Reduction:          36%


> that 36% reduction is because the lm_head and embedding layer are ignored 

In [31]:
## testing originla model 
prompt = "Machine learning is a branch of"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = base_model.generate(
    **inputs, 
    max_new_tokens=60, 
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Base Model ({MODEL_NAME})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Base Model (Qwen/Qwen3-0.6B)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has gained significant attention in recent years, particularly in the context of the rise of big data and the need for efficient, scalable solutions to complex problems. As the field continues to evolve, the integration of machine learning into various industries is becoming increasingly widespread. However, despite its growing popularity,


In [34]:
## testing quantized model 
prompt = "Machine learning is branch of"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
quantized_model = AutoModelForCausalLM.from_pretrained(
    f"/kaggle/working/{OUTPUT_DIR}",device_map="cpu",dtype=torch.bfloat16,
)
inputs = tokenizer(prompt,return_tensors="pt")
outputs = quantized_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Quantized Model ({MODEL_NAME})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Compressing model: 100%|██████████| 196/196 [00:00<00:00, 491.25it/s]


Loading weights:   0%|          | 0/506 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 196/196 [00:00<00:00, 287.79it/s]


Quantized Model (Qwen/Qwen3-0.6B)
Prompt: Machine learning is branch of
Response:  computer science, which is used to develop algorithms and models that can perform tasks that would be very difficult for a computer to perform. It is used in various fields such as healthcare, finance, and more. The main goal of this project is to develop a machine learning model that can predict the probability of


In [36]:
## perplezity compraion 
from datasets import load_dataset

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    ## here fist all the text (lines are joined)
    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    ## an sliding window is created => to predict the next sequence 
    ## here the predicted and actual prob distribution is trying and that express the uncertanity/suprise
    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

test_data = load_dataset(
    "Salesforce/wikitext", 
    "wikitext-2-raw-v1", 
    split="test"
)
print(f"Loaded {len(test_data)} test samples")

Loaded 4358 test samples


In [37]:
quant_ppl = calculate_perplexity(quantized_model, tokenizer, test_data)
print(f"Quantized perplexity: {quant_ppl:.2f}")

Quantized perplexity: 33.16


In [39]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="cpu", dtype=torch.bfloat16,
)
base_ppl = calculate_perplexity(base_model, tokenizer, test_data)
print(f"Base perplexity: {base_ppl:.2f}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Base perplexity: 32.78


In [41]:
print("Perplexity Comparison")
print("=" * 40)
print(f"Base (BF16):      {base_ppl:.2f}")
print(f"Quantized (W4A16): {quant_ppl:.2f}")
print(f"Difference:       {quant_ppl - base_ppl:+.2f} ({(quant_ppl/base_ppl - 1)*100:+.1f}%)")
print(f"\nA small increase in perplexity is expected — the quantized layers use 8-bit weights.")

Perplexity Comparison
Base (BF16):      32.78
Quantized (W4A16): 33.16
Difference:       +0.38 (+1.2%)

A small increase in perplexity is expected — the quantized layers use 8-bit weights.


## creating wrapper in class 

In [42]:
import os
import math
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot


class Quantize:
    def __init__(self, recipe, model_name, output_dir="Outputs"):
        self.recipe = recipe
        self.model_name = model_name
        self.output_dir = output_dir

        # Tokenizer setup
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Lazy model references
        self.base_model = None
        self.quantized_model = None

        # Load datasets
        self.train_dataset = load_dataset(
            "Salesforce/wikitext",
            "wikitext-2-raw-v1",
            split="train",
        )
        self.test_dataset = load_dataset(
            "Salesforce/wikitext",
            "wikitext-2-raw-v1",
            split="test",
        )

    def load_base_model(self, device="cpu"):
        if self.base_model is None:
            self.base_model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                device_map=device,
                torch_dtype=torch.bfloat16,
            )
        return self.base_model

    def quantize_model(self, num_calibration_samples=256, max_seq_length=2048):
        save_path = f"/kaggle/working/{self.output_dir}"

        # 1. Filter empty lines
        clean_dataset = self.train_dataset.filter(lambda x: len(x["text"].strip()) > 0)

        # 2. Tokenize calibration data
        def tokenize_fn(examples):
            return self.tokenizer(
                examples["text"],
                truncation=True,
                max_length=max_seq_length,
                return_tensors=None,
            )

        tokenized_dataset = clean_dataset.map(
            tokenize_fn,
            batched=True,
            remove_columns=clean_dataset.column_names,
        )

        # 3. Enforce torch int64 tensors
        tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

        # 4. Run quantization if not already saved
        if not os.path.isdir(save_path):
            oneshot(
                model=self.model_name,
                dataset=tokenized_dataset,
                recipe=self.recipe,
                output_dir=self.output_dir,
                max_seq_length=max_seq_length,
                num_calibration_samples=num_calibration_samples,
            )
            print(f"Quantization complete. Model saved to: {save_path}")
        else:
            print(f"Quantized model already exists at: {save_path}")

    def load_quantized_model(self, device="cpu"):
        save_path = f"/kaggle/working/{self.output_dir}"
        if self.quantized_model is None:
            self.quantized_model = AutoModelForCausalLM.from_pretrained(
                save_path,
                device_map=device,
                torch_dtype=torch.bfloat16,
            )
        return self.quantized_model

    def calculate_perplexity(self, model, max_tokens=5000, stride=512):
        encodings = self.tokenizer(
            "\n\n".join(self.test_dataset["text"]),
            return_tensors="pt",
            truncation=True,
            max_length=max_tokens,
        )
        input_ids = encodings.input_ids.to(model.device)
        nlls, prev_end = [], 0

        for begin_loc in range(0, input_ids.size(1), stride):
            end_loc = min(begin_loc + stride, input_ids.size(1))
            trg_len = end_loc - prev_end
            input_slice = input_ids[:, begin_loc:end_loc]
            target_slice = input_slice.clone()
            target_slice[:, :-trg_len] = -100

            with torch.no_grad():
                loss = model(input_slice, labels=target_slice).loss
                nlls.append(loss * trg_len)
            prev_end = end_loc

        return math.exp(torch.stack(nlls).sum().item() / prev_end)

    def test_inference(self, prompt, is_quantized=False, max_new_tokens=60):
        target_model = self.load_quantized_model() if is_quantized else self.load_base_model()

        inputs = self.tokenizer(prompt, return_tensors="pt").to(target_model.device)
        with torch.no_grad():
            outputs = target_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[-1] :]
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [45]:
from llmcompressor.modifiers.transform.awq import AWQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

# AWQ is typically used as a pre-quantization transform recipe alongside the standard QuantizationModifier
recipe = [
    AWQModifier(duo_scaling="both"),
    QuantizationModifier(
        scheme="W8A8", 
        targets="Linear", 
        ignore=["lm_head"]
    )
]

In [ ]:
pipeline = Quantize(recipe=recipe, model_name=MODEL_NAME, output_dir=f"{OUTPUT_DIR}/AWQ")

# 1. Run quantization
pipeline.quantize_model()

# 2. Test generation
print("Base:", pipeline.test_inference("Machine learning is", is_quantized=False))
print("Quant:", pipeline.test_inference("Machine learning is", is_quantized=True))

# 3. Compare Perplexity
base_ppl = pipeline.calculate_perplexity(pipeline.load_base_model())
quant_ppl = pipeline.calculate_perplexity(pipeline.load_quantized_model())
print(f"Base PPL: {base_ppl:.2f} | Quantized PPL: {quant_ppl:.2f}")

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

2026-10-23T05:25:39.8606 | reset | INFO - Compression lifecycle reset
2026-10-23T05:25:39.8669 | from_modifiers | INFO - Creating recipe from modifiers
2026-10-23T05:25:39.8678 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Applying quantization config: 100%|██████████| 196/196 [00:00<00:00, 6861.56it/s]

2026-10-23T05:25:39.9075 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-10-23T05:25:39.9083 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`


2026-10-23T05:25:40.4414 | _set_resolved_mappings | WARNING - 28 mappings were skipped due to incompatible shapes. Set logging to DEBUG to view full list.


(1/29): Calibrating: 100%|██████████| 256/256 [00:00<00:00, 833.80it/s]
Smoothing: 0it [00:00, ?it/s]
Smoothing:  33%|███▎      | 1/3 [00:08<00:16,  8.46s/it]                                                             
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:00<?, ?it/s]
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:00<?, ?it/s, best_error=2.214e-06]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:00<00:06,  2.87it/s, best_error=2.214e-06]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:00<00:06,  2.87it/s, best_error=1.981e-06]
Grid search for model.layers.0.post_attention_layernorm:  10%|█         | 2/20 [00:00<00:06,  2.87it/s, best_error=1.981e-06]
Grid search for model.layers.0.post_attention_layernorm:  10%|█         | 2/20 [00:01<00:06,  2.87it/s, best_error=1.981e-06]
Grid search for model.layers.0.post_attention_layernorm:  15%|